# AI Honeypot + Real-Time Threat Dashboard

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import random
import threading
import time
import os
from flask import Flask, render_template, jsonify
from scapy.all import sniff, IP
import logging
import signal
import sys

# Initialize Flask app
app = Flask(__name__)

# Data structures
df_log = []
blocked_ips = set()

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("IDS")

# Flag to control background threads
running = True

# Honeypot service (mimicking different services)
def honeypot_service():
    return random.choice(['OpenSSH_8.2p1 Ubuntu', 'nginx/1.18.0', 'Apache/2.4.41', 'PostgreSQL 13.2'])

@app.route('/')
def home():
    client_ip = request.remote_addr
    logger.info(f"Connection from: {client_ip}")
    return honeypot_service()

@app.route('/dashboard')
def dashboard():
    try:
        # Simple stats: count of malicious and benign traffic
        malicious = len([entry for entry in df_log if entry['malicious'] == 1])
        benign = len(df_log) - malicious

        # Count of blocked IPs
        blocked = len(blocked_ips)

        return render_template("dashboard.html", malicious=malicious, benign=benign, blocked=blocked)
    except Exception as e:
        logger.error(f"Dashboard error: {e}")
        return f"Error generating dashboard: {str(e)}", 500

@app.route('/api/stats')
def stats():
    return jsonify({
        'total_packets': len(df_log),
        'malicious_packets': len([entry for entry in df_log if entry['malicious'] == 1]),
        'benign_packets': len([entry for entry in df_log if entry['malicious'] == 0]),
        'blocked_ips': list(blocked_ips)
    })

def capture(pkt):
    if IP in pkt:
        src = pkt[IP].src
        dst = pkt[IP].dst
        proto = pkt[IP].proto
        length = len(pkt)

        # Classify the packet: 20% chance of malicious traffic for simplicity
        label = 1 if random.random() < 0.2 else 0

        # Store the packet info
        df_log.append({
            'src': src,
            'dst': dst,
            'proto': proto,
            'length': length,
            'malicious': label
        })

        # Block malicious IPs (just an example, doesn't actually block)
        if label == 1:
            blocked_ips.add(src)
            logger.warning(f"Malicious traffic detected from {src}")

def blocking_loop():
    while running:  # Run while the flag is True
        for ip in blocked_ips:
            logger.info(f"Blocked IP: {ip}")
        time.sleep(30)

# Start sniffing packets and running blocking loop in separate threads
def start_threads():
    threading.Thread(target=lambda: sniff(prn=capture, store=0), daemon=True).start()
    threading.Thread(target=blocking_loop, daemon=True).start()

def shutdown_gracefully(signal, frame):
    global running
    running = False
    logger.info("Shutting down gracefully...")
    sys.exit(0)  # Gracefully exit the program

# Register signal handler for graceful shutdown
signal.signal(signal.SIGINT, shutdown_gracefully)

if __name__ == "__main__":
    # Ensure the static folder exists for dashboard rendering
    if not os.path.exists('templates'):
        os.makedirs('templates')

    # Start threads for packet sniffing and IP blocking
    start_threads()

    # Run the Flask app
    logger.info("IDS started")
    app.run(host='0.0.0.0', port=5000, debug=False)
